# Feature Detectors and Descriptors

This notebook is designed as a self-learning chapter for computer vision students. It follows the learning style of a concept-first educational notebook: each section starts with the core idea, then gives runnable Python code, then asks students to complete or interpret a task.

**Learning goals**

By the end of this notebook, you should be able to:

1. Explain why feature descriptors are needed after detecting feature points.
2. Compare raw-pixel, gradient-based, histogram-based, and spatial-histogram descriptors.
3. Implement simple versions of Harris corners, MOPS, integral images, GIST-like descriptors, texton histograms, HOG, SURF-style summaries, and SIFT-style descriptors.
4. Match descriptors and evaluate how invariance and discriminative power trade off.

**Notebook convention**

All formulas use Jupyter-stable inline math `$...$` and display math `$$...$$`.

## 0. Setup and Helper Utilities

The code uses common scientific Python libraries. The notebook is intentionally written with lightweight dependencies. If `scikit-image` or `scikit-learn` is unavailable, most sections still run with synthetic images and NumPy-based implementations.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from scipy import ndimage as ndi

try:
    from skimage import data, color, transform, util
    HAS_SKIMAGE = True
except Exception:
    HAS_SKIMAGE = False

try:
    from sklearn.cluster import KMeans
    HAS_SKLEARN = True
except Exception:
    HAS_SKLEARN = False

np.set_printoptions(precision=3, suppress=True)


def show_image(img, title="Image", cmap="gray"):
    """Display one image with a clean layout."""
    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()


def normalize01(x):
    """Normalize an array to the range [0, 1] for visualization."""
    x = np.asarray(x, dtype=float)
    mn, mx = np.min(x), np.max(x)
    if mx - mn < 1e-12:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)


def make_synthetic_image(size=256):
    """Create a synthetic image with corners, edges, blobs, and texture."""
    yy, xx = np.mgrid[0:size, 0:size]
    img = np.zeros((size, size), dtype=float)
    img[35:120, 35:120] = 0.85
    img[140:215, 45:130] = 0.55
    circle = (xx - 180) ** 2 + (yy - 80) ** 2 < 38 ** 2
    img[circle] = 0.75
    stripe = ((xx + 2 * yy) % 24) < 10
    img[150:235, 150:235] = 0.25 + 0.45 * stripe[150:235, 150:235]
    img += 0.05 * np.random.default_rng(0).normal(size=img.shape)
    return np.clip(img, 0, 1)


def load_demo_image():
    """Load a demo grayscale image, or create a synthetic one if needed."""
    if HAS_SKIMAGE:
        img = data.camera().astype(float) / 255.0
        img = transform.resize(img, (256, 256), anti_aliasing=True)
        return img
    return make_synthetic_image(256)

img = load_demo_image()
show_image(img, "Demo image")

## 1. Why Feature Descriptors?

A detector tells us **where** an interesting point is. A descriptor tells us **what the local image region looks like** so that we can match it across images.

In many computer vision tasks, the same physical point can appear under different conditions:

- **Photometric changes:** brightness, contrast, exposure, illumination color.
- **Geometric changes:** translation, rotation, scale, viewpoint deformation.
- **Imaging changes:** blur, noise, compression, sensor differences.

A useful descriptor should be:

- **Repeatable:** similar for the same real-world point across images.
- **Distinctive:** different for different points.
- **Efficient:** fast to compute and compact to store.

A basic matching objective compares two descriptors $d_i$ and $d_j$ by distance:

$$
D(d_i, d_j) = \lVert d_i - d_j \rVert_2
$$

Smaller distance usually means a better match, but only if the descriptor has the right invariances.

**Task 1.1**  
Before running code, write down one example where raw pixel patches would fail, and one example where they would work well.

### Code: Raw Patch Matching Under Brightness Changes

Raw pixel descriptors are simple: flatten a local image patch into a vector. They are sensitive to brightness and contrast, so we also test normalized patches.

In [ ]:
def extract_patch(image, center, radius=16):
    """Extract a square patch centered at (row, col)."""
    r, c = center
    r0, r1 = r - radius, r + radius
    c0, c1 = c - radius, c + radius
    if r0 < 0 or c0 < 0 or r1 > image.shape[0] or c1 > image.shape[1]:
        raise ValueError("Patch goes outside image boundary.")
    return image[r0:r1, c0:c1].copy()


def raw_descriptor(patch):
    """Flatten raw pixel values into a descriptor vector."""
    return patch.astype(float).ravel()


def normalized_patch_descriptor(patch, eps=1e-8):
    """Normalize a patch to reduce sensitivity to brightness offset and contrast gain."""
    p = patch.astype(float)
    return ((p - p.mean()) / (p.std() + eps)).ravel()

patch_a = extract_patch(img, (95, 95), radius=24)
patch_b = np.clip(1.4 * patch_a + 0.2, 0, 1)

raw_dist = np.linalg.norm(raw_descriptor(patch_a) - raw_descriptor(patch_b))
norm_dist = np.linalg.norm(normalized_patch_descriptor(patch_a) - normalized_patch_descriptor(patch_b))

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(patch_a, cmap="gray"); ax[0].set_title("Original patch"); ax[0].axis("off")
ax[1].imshow(patch_b, cmap="gray"); ax[1].set_title("Brightness changed"); ax[1].axis("off")
plt.show()

print(f"Raw descriptor distance: {raw_dist:.3f}")
print(f"Normalized patch descriptor distance: {norm_dist:.3f}")

## 2. Image Gradients

Gradients describe local intensity change. They reduce sensitivity to absolute intensity because they use differences rather than raw brightness values.

For image $I(x, y)$, horizontal and vertical derivatives are:

$$
I_x = \frac{\partial I}{\partial x}, \qquad I_y = \frac{\partial I}{\partial y}
$$

Gradient magnitude and orientation are:

$$
M = \sqrt{I_x^2 + I_y^2}
$$

$$
\theta = \operatorname{atan2}(I_y, I_x)
$$

**Task 2.1**  
Which image structures produce strong gradient magnitude: flat regions, edges, corners, or all of them?

### Code: Compute and Visualize Gradients

In [ ]:
def image_gradients(image, sigma=1.0):
    """Compute smoothed image gradients using Gaussian derivatives."""
    smooth = ndi.gaussian_filter(image, sigma=sigma)
    iy, ix = np.gradient(smooth)
    mag = np.sqrt(ix ** 2 + iy ** 2)
    ori = np.arctan2(iy, ix)
    return ix, iy, mag, ori

ix, iy, mag, ori = image_gradients(img, sigma=1.0)

show_image(normalize01(mag), "Gradient magnitude")
show_image(ori, "Gradient orientation", cmap="twilight")

## 3. Harris Corner Detector

A corner is a point where intensity changes strongly in more than one direction. Harris uses the second-moment matrix:

$$
M = \sum_{(x,y) \in W}
w(x,y)
\begin{bmatrix}
I_x^2 & I_x I_y \\
I_x I_y & I_y^2
\end{bmatrix}
$$

The Harris response is:

$$
R = \det(M) - k \operatorname{trace}(M)^2
$$

A large positive $R$ suggests a corner. A large negative $R$ often suggests an edge.

**Task 3.1**  
Change `k` and `threshold_rel`. Observe whether you get more edge-like detections or cleaner corner detections.

### Code: Harris Response and Non-Maximum Suppression

In [ ]:
def harris_response(image, sigma_grad=1.0, sigma_window=2.0, k=0.04):
    """Compute Harris corner response."""
    ix, iy, _, _ = image_gradients(image, sigma=sigma_grad)
    ixx = ndi.gaussian_filter(ix * ix, sigma_window)
    iyy = ndi.gaussian_filter(iy * iy, sigma_window)
    ixy = ndi.gaussian_filter(ix * iy, sigma_window)
    det = ixx * iyy - ixy ** 2
    trace = ixx + iyy
    return det - k * trace ** 2


def detect_harris_corners(image, threshold_rel=0.02, min_distance=8):
    """Detect Harris corners with simple maximum filtering."""
    R = harris_response(image)
    threshold = threshold_rel * R.max()
    local_max = R == ndi.maximum_filter(R, size=2 * min_distance + 1)
    corners = np.argwhere((R > threshold) & local_max)
    return R, corners

R, corners = detect_harris_corners(img, threshold_rel=0.02, min_distance=8)

plt.figure(figsize=(6, 6))
plt.imshow(img, cmap="gray")
plt.scatter(corners[:, 1], corners[:, 0], s=20, facecolors="none", edgecolors="r")
plt.title(f"Harris corners: {len(corners)} detected")
plt.axis("off")
plt.show()

## 4. Multi-Scale Detection

A feature can appear at different sizes. Multi-scale detection searches over both position and scale. A common idea is to build a scale-space by smoothing the image with Gaussians of increasing $\sigma$:

$$
L(x, y, \sigma) = G(x, y, \sigma) * I(x, y)
$$

The Difference of Gaussians approximation is:

$$
D(x, y, \sigma) = L(x, y, k\sigma) - L(x, y, \sigma)
$$

SIFT detects extrema in this scale-space. In this notebook, we use a simplified visualization rather than a full detector.

**Task 4.1**  
Look at the responses for different scales. Which scale responds more strongly to large structures?

### Code: Difference of Gaussians Across Scales

In [ ]:
def dog_pyramid(image, sigmas=(1, 2, 4, 8), k=1.6):
    """Compute a small Difference-of-Gaussians pyramid."""
    dogs = []
    for sigma in sigmas:
        low = ndi.gaussian_filter(image, sigma)
        high = ndi.gaussian_filter(image, sigma * k)
        dogs.append(high - low)
    return dogs

dogs = dog_pyramid(img)

for i, dog in enumerate(dogs):
    show_image(normalize01(np.abs(dog)), f"Absolute DoG response at scale {i}")

## 5. Orientation Normalization

To make a patch descriptor more rotation-tolerant, estimate a dominant orientation and rotate the patch to a canonical direction.

A simple orientation estimate uses gradient magnitudes as votes:

$$
\theta^* = \arg\max_\theta \sum_{p \in \text{patch}} M(p) \cdot \mathbf{1}[\theta(p) \in \text{bin}(\theta)]
$$

This is a key idea behind MOPS and SIFT-style descriptors.

**Task 5.1**  
Rotate the input patch by different angles. Does orientation normalization recover a similar canonical patch every time?

### Code: Dominant Orientation and Patch Rotation

In [ ]:
def dominant_orientation(patch, bins=36):
    """Estimate dominant gradient orientation in a patch."""
    _, _, mag, ori = image_gradients(patch, sigma=1.0)
    ori_deg = (np.rad2deg(ori) + 360) % 360
    hist, edges = np.histogram(ori_deg, bins=bins, range=(0, 360), weights=mag)
    best = np.argmax(hist)
    return 0.5 * (edges[best] + edges[best + 1])


def rotate_patch_to_canonical(patch):
    """Rotate patch so its dominant orientation points to zero degrees."""
    angle = dominant_orientation(patch)
    rotated = ndi.rotate(patch, -angle, reshape=False, order=1, mode="nearest")
    return rotated, angle

patch = extract_patch(img, (95, 95), radius=32)
rotated_patch = ndi.rotate(patch, 35, reshape=False, order=1, mode="nearest")
canon, angle = rotate_patch_to_canonical(rotated_patch)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(patch, cmap="gray"); ax[0].set_title("Original patch"); ax[0].axis("off")
ax[1].imshow(rotated_patch, cmap="gray"); ax[1].set_title("Rotated patch"); ax[1].axis("off")
ax[2].imshow(canon, cmap="gray"); ax[2].set_title(f"Canonical patch\nangle={angle:.1f} deg"); ax[2].axis("off")
plt.show()

## 6. MOPS: Multi-Scale Oriented Patches

MOPS represents a keypoint by a normalized, low-resolution patch. The lecture pipeline is:

1. Extract a $40 \times 40$ oriented image patch.
2. Subsample every 5 pixels to get an $8 \times 8$ descriptor.
3. Subtract the mean and divide by standard deviation.
4. Optionally project with Haar wavelet-like filters.

The normalization step reduces bias and gain:

$$
\hat{x}_i = \frac{x_i - \mu}{\sigma + \epsilon}
$$

MOPS is less detailed than SIFT, but it is simple and fast.

**Task 6.1**  
Compare a raw $8 \times 8$ downsampled patch with a normalized MOPS descriptor. Which is more robust to brightness changes?

### Code: MOPS Descriptor

In [ ]:
def mops_descriptor(image, center, radius=20, output_size=8, normalize=True):
    """Compute a simplified MOPS descriptor from a local patch."""
    patch = extract_patch(image, center, radius=radius)
    canonical, angle = rotate_patch_to_canonical(patch)
    zoom = output_size / canonical.shape[0]
    small = ndi.zoom(canonical, zoom=zoom, order=1)
    small = small[:output_size, :output_size]
    desc = small.astype(float).ravel()
    if normalize:
        desc = (desc - desc.mean()) / (desc.std() + 1e-8)
    return desc, small, angle

center = tuple(corners[0]) if len(corners) else (95, 95)
desc, small_patch, angle = mops_descriptor(img, center)

show_image(small_patch, f"MOPS 8x8 patch at {center}, angle={angle:.1f} deg")
print("Descriptor shape:", desc.shape)
print("First 10 descriptor values:", desc[:10])

## 7. Integral Images and Haar-Like Features

An integral image stores cumulative sums so that any rectangular region can be summed quickly.

For image $I$, the integral image $S$ is:

$$
S(x, y) = \sum_{i \leq x, j \leq y} I(i, j)
$$

The sum of a rectangle can be computed with four array accesses:

$$
\operatorname{sum}(A) = S(D) - S(B) - S(C) + S(A)
$$

Haar-like features use differences of rectangle sums. This is important for SURF-style descriptors because Haar responses can be computed efficiently.

**Task 7.1**  
Use the small matrix example and verify the bottom-right $2 \times 2$ sum manually.

### Code: Integral Image and Rectangle Sum

In [ ]:
def integral_image(image):
    """Compute an integral image with one-pixel zero padding."""
    return np.pad(image, ((1, 0), (1, 0)), mode="constant").cumsum(axis=0).cumsum(axis=1)


def rect_sum(ii, top, left, bottom, right):
    """Return sum over image[top:bottom, left:right] using a padded integral image."""
    return ii[bottom, right] - ii[top, right] - ii[bottom, left] + ii[top, left]

small = np.array([[1, 5, 2],
                  [2, 4, 1],
                  [2, 1, 1]], dtype=float)
ii = integral_image(small)

print("Original image:")
print(small)
print("Padded integral image:")
print(ii)
print("Bottom-right 2x2 sum:", rect_sum(ii, 1, 1, 3, 3))

## 8. GIST-Like Descriptor

GIST encodes the rough spatial distribution of image gradients and filter responses. A simplified version:

1. Apply a filter bank, such as oriented Gabor filters.
2. Split the image into a $4 \times 4$ grid.
3. Average each filter response in each cell.
4. Concatenate the results.

If the filter bank has $N$ filters, the descriptor size is:

$$
4 \times 4 \times N
$$

GIST is more global than local descriptors such as SIFT.

**Task 8.1**  
Try changing the number of orientations. How does the descriptor length change?

### Code: Simple Gabor Filter Bank and GIST-Like Descriptor

In [ ]:
def gabor_kernel(size=31, sigma=4.0, theta=0.0, frequency=0.15):
    """Create a real-valued Gabor-like kernel."""
    ax = np.arange(-(size // 2), size // 2 + 1)
    yy, xx = np.meshgrid(ax, ax)
    x_theta = xx * np.cos(theta) + yy * np.sin(theta)
    y_theta = -xx * np.sin(theta) + yy * np.cos(theta)
    envelope = np.exp(-(x_theta ** 2 + y_theta ** 2) / (2 * sigma ** 2))
    carrier = np.cos(2 * np.pi * frequency * x_theta)
    kernel = envelope * carrier
    return kernel - kernel.mean()


def gist_like_descriptor(image, grid=4, orientations=8):
    """Compute a simplified GIST-like descriptor using oriented Gabor filters."""
    responses = []
    for k in range(orientations):
        theta = np.pi * k / orientations
        kernel = gabor_kernel(theta=theta)
        response = ndi.convolve(image, kernel, mode="reflect")
        responses.append(np.abs(response))

    h, w = image.shape
    cell_h, cell_w = h // grid, w // grid
    desc = []
    for response in responses:
        for gy in range(grid):
            for gx in range(grid):
                cell = response[gy * cell_h:(gy + 1) * cell_h, gx * cell_w:(gx + 1) * cell_w]
                desc.append(cell.mean())
    return np.array(desc), responses

gist_desc, responses = gist_like_descriptor(img, grid=4, orientations=8)
print("GIST-like descriptor length:", len(gist_desc))
show_image(normalize01(responses[0]), "Example Gabor response")

## 9. Histogram of Textons

Textons are learned texture primitives. A typical texton pipeline is:

1. Compute filter-bank responses for many pixels or patches.
2. Cluster these responses to form a texton dictionary.
3. Assign each pixel or patch to its nearest texton.
4. Pool the assignments into a histogram.

This follows the pattern:

$$
\text{image} \rightarrow \text{filter responses} \rightarrow \text{texton map} \rightarrow \text{histogram}
$$

A texton histogram may ignore exact spatial layout, which is useful for stochastic textures.

**Task 9.1**  
Run the texton code on the synthetic image and on the camera image. Which one has clearer texture clusters?

### Code: Learn a Small Texton Dictionary

In [ ]:
def local_filter_features(image):
    """Build per-pixel features from a small filter bank."""
    ix, iy, mag, _ = image_gradients(image, sigma=1.0)
    lap = ndi.gaussian_laplace(image, sigma=2.0)
    blur = ndi.gaussian_filter(image, sigma=3.0)
    feats = np.stack([image, ix, iy, mag, lap, blur], axis=-1)
    return feats


def texton_histogram(image, n_textons=8, sample_count=4000):
    """Learn textons from sampled filter responses and return a texton histogram."""
    feats = local_filter_features(image)
    flat = feats.reshape(-1, feats.shape[-1])

    rng = np.random.default_rng(1)
    idx = rng.choice(len(flat), size=min(sample_count, len(flat)), replace=False)
    samples = flat[idx]

    if not HAS_SKLEARN:
        raise ImportError("scikit-learn is required for KMeans in this section.")

    km = KMeans(n_clusters=n_textons, random_state=0, n_init=10)
    km.fit(samples)
    labels = km.predict(flat).reshape(image.shape)
    hist = np.bincount(labels.ravel(), minlength=n_textons).astype(float)
    hist /= hist.sum()
    return labels, hist

if HAS_SKLEARN:
    labels, hist = texton_histogram(make_synthetic_image(160), n_textons=8)
    show_image(labels, "Texton map", cmap="tab20")
    plt.figure(figsize=(6, 3))
    plt.bar(np.arange(len(hist)), hist)
    plt.title("Texton histogram")
    plt.xlabel("Texton index")
    plt.ylabel("Frequency")
    plt.show()
else:
    print("Install scikit-learn to run this section.")

## 10. HOG: Histogram of Oriented Gradients

HOG describes local shape using histograms of gradient orientations. A common setup uses:

- Cells of $8 \times 8$ pixels.
- Blocks of $2 \times 2$ cells.
- Unsigned gradient orientations from $0^\circ$ to $180^\circ$.
- L2 normalization per block.

For one block vector $v$, L2 normalization is:

$$
\hat{v} = \frac{v}{\sqrt{\lVert v \rVert_2^2 + \epsilon^2}}
$$

For a $64 \times 128$ pedestrian window with $8 \times 8$ cells, there are $8 \times 16$ cells and $7 \times 15$ overlapping blocks. If each cell has 9 bins, each block has $2 \times 2 \times 9 = 36$ values, so the descriptor has:

$$
7 \times 15 \times 36 = 3780
$$

**Task 10.1**  
Explain why overlapping blocks create redundancy. How many times does an inner cell appear in different blocks?

### Code: HOG From Scratch

In [ ]:
def hog_descriptor(image, cell_size=8, block_size=2, bins=9, eps=1e-6):
    """Compute a simple HOG descriptor with unsigned orientations."""
    ix, iy, mag, ori = image_gradients(image, sigma=0.5)
    angles = (np.rad2deg(ori) + 180) % 180

    h, w = image.shape
    ncy, ncx = h // cell_size, w // cell_size
    hist = np.zeros((ncy, ncx, bins), dtype=float)
    bin_width = 180 / bins

    for cy in range(ncy):
        for cx in range(ncx):
            ys = slice(cy * cell_size, (cy + 1) * cell_size)
            xs = slice(cx * cell_size, (cx + 1) * cell_size)
            cell_angles = angles[ys, xs].ravel()
            cell_mag = mag[ys, xs].ravel()
            bin_idx = np.floor(cell_angles / bin_width).astype(int) % bins
            for b, m in zip(bin_idx, cell_mag):
                hist[cy, cx, b] += m

    blocks = []
    for by in range(ncy - block_size + 1):
        for bx in range(ncx - block_size + 1):
            block = hist[by:by + block_size, bx:bx + block_size, :].ravel()
            block = block / np.sqrt(np.sum(block ** 2) + eps ** 2)
            blocks.append(block)
    return np.concatenate(blocks), hist

# Resize or crop to a pedestrian-like 64x128 window for the dimensionality check.
window = img[:128, :64]
hog_desc, cell_hist = hog_descriptor(window, cell_size=8, block_size=2, bins=9)
print("HOG descriptor length:", len(hog_desc))
print("Expected length:", 7 * 15 * 36)

plt.figure(figsize=(6, 3))
plt.bar(np.arange(9), cell_hist[cell_hist.shape[0] // 2, cell_hist.shape[1] // 2])
plt.title("Example HOG cell histogram")
plt.xlabel("Orientation bin")
plt.ylabel("Weighted count")
plt.show()

## 11. SURF-Style Descriptor

SURF uses Haar wavelet responses in a local region. A simplified SURF-style descriptor divides the neighborhood into a $4 \times 4$ grid. Each cell is represented by four values:

$$
\left[ \sum d_x, \sum d_y, \sum |d_x|, \sum |d_y| \right]
$$

The descriptor length is therefore:

$$
4 \times 4 \times 4 = 64
$$

SURF is designed for efficiency: Haar responses can be computed quickly with integral images.

**Task 11.1**  
Why do both signed sums and absolute sums appear in the descriptor?

### Code: Simplified SURF-Style Descriptor

In [ ]:
def surf_style_descriptor(image, center, radius=32, grid=4):
    """Compute a simplified SURF-style descriptor using image derivatives."""
    patch = extract_patch(image, center, radius=radius)
    dx, dy, _, _ = image_gradients(patch, sigma=1.0)
    h, w = patch.shape
    cell_h, cell_w = h // grid, w // grid

    desc = []
    for gy in range(grid):
        for gx in range(grid):
            ys = slice(gy * cell_h, (gy + 1) * cell_h)
            xs = slice(gx * cell_w, (gx + 1) * cell_w)
            cx = dx[ys, xs]
            cy = dy[ys, xs]
            desc.extend([cx.sum(), cy.sum(), np.abs(cx).sum(), np.abs(cy).sum()])

    desc = np.array(desc, dtype=float)
    desc = desc / (np.linalg.norm(desc) + 1e-8)
    return desc

safe_center = (128, 128)
surf_desc = surf_style_descriptor(img, safe_center, radius=32)
print("SURF-style descriptor length:", len(surf_desc))
print("First 12 values:", surf_desc[:12])

## 12. SIFT-Style Descriptor

SIFT includes both a detector and a descriptor. The full SIFT pipeline contains:

1. Multi-scale extrema detection.
2. Keypoint localization.
3. Orientation assignment.
4. Descriptor construction.

The descriptor uses a $4 \times 4$ grid of cells. Each cell stores an 8-bin orientation histogram, so the descriptor size is:

$$
16 \times 8 = 128
$$

SIFT uses Gaussian weighting near the center of the patch, soft binning, normalization, clipping, and renormalization. Here we implement a simplified version.

**Task 12.1**  
Compare the SIFT-style descriptor with MOPS. Which one keeps more local spatial and orientation information?

### Code: Simplified SIFT-Style Descriptor

In [ ]:
def sift_style_descriptor(image, center, radius=32, grid=4, bins=8):
    """Compute a simplified SIFT-style descriptor."""
    patch = extract_patch(image, center, radius=radius)
    canonical, angle = rotate_patch_to_canonical(patch)
    ix, iy, mag, ori = image_gradients(canonical, sigma=1.0)

    h, w = canonical.shape
    yy, xx = np.mgrid[0:h, 0:w]
    cy, cx = (h - 1) / 2, (w - 1) / 2
    sigma = 0.5 * max(h, w)
    weight = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * sigma ** 2))
    mag = mag * weight

    angles = (np.rad2deg(ori) + 360) % 360
    cell_h, cell_w = h // grid, w // grid
    desc = []
    bin_width = 360 / bins

    for gy in range(grid):
        for gx in range(grid):
            ys = slice(gy * cell_h, (gy + 1) * cell_h)
            xs = slice(gx * cell_w, (gx + 1) * cell_w)
            hist = np.zeros(bins, dtype=float)
            cell_angles = angles[ys, xs].ravel()
            cell_mag = mag[ys, xs].ravel()
            bin_idx = np.floor(cell_angles / bin_width).astype(int) % bins
            for b, m in zip(bin_idx, cell_mag):
                hist[b] += m
            desc.extend(hist)

    desc = np.array(desc, dtype=float)
    desc = desc / (np.linalg.norm(desc) + 1e-8)
    desc = np.clip(desc, 0, 0.2)
    desc = desc / (np.linalg.norm(desc) + 1e-8)
    return desc, angle

sift_desc, sift_angle = sift_style_descriptor(img, safe_center)
print("SIFT-style descriptor length:", len(sift_desc))
print("Assigned orientation:", sift_angle)
print("First 12 values:", sift_desc[:12])

## 13. Descriptor Matching and Lowe Ratio Test

After descriptors are computed, matching can be done by nearest-neighbor search. However, the nearest neighbor is not always reliable. SIFT popularized the ratio test:

$$
\frac{D_1}{D_2} < \tau
$$

Here $D_1$ is the distance to the nearest descriptor and $D_2$ is the distance to the second-nearest descriptor. A low ratio means the best match is much better than the runner-up.

**Task 13.1**  
Try thresholds $\tau = 0.6$, $0.75$, and $0.9$. Which threshold gives more matches? Which is likely more reliable?

### Code: Match MOPS Descriptors Across a Transformed Image

In [ ]:
def transform_image(image, angle=12, shift=(6, -4), brightness=1.1):
    """Apply a simple geometric and photometric transform."""
    rotated = ndi.rotate(image, angle, reshape=False, order=1, mode="reflect")
    shifted = ndi.shift(rotated, shift=shift, order=1, mode="reflect")
    return np.clip(brightness * shifted, 0, 1)


def pairwise_distances(A, B):
    """Compute Euclidean distances between two descriptor matrices."""
    return np.sqrt(((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=2))


def ratio_match(desc_a, desc_b, ratio=0.75):
    """Return descriptor matches that pass the nearest-neighbor ratio test."""
    D = pairwise_distances(desc_a, desc_b)
    order = np.argsort(D, axis=1)
    matches = []
    for i in range(D.shape[0]):
        j1, j2 = order[i, 0], order[i, 1]
        if D[i, j1] / (D[i, j2] + 1e-8) < ratio:
            matches.append((i, j1, D[i, j1], D[i, j2]))
    return matches

img2 = transform_image(img)
_, corners1 = detect_harris_corners(img, threshold_rel=0.02, min_distance=12)
_, corners2 = detect_harris_corners(img2, threshold_rel=0.02, min_distance=12)

# Keep only corners where a descriptor patch can fit safely.
def keep_safe(corners, image_shape, radius=24, limit=60):
    safe = []
    for r, c in corners:
        if radius <= r < image_shape[0] - radius and radius <= c < image_shape[1] - radius:
            safe.append((int(r), int(c)))
        if len(safe) >= limit:
            break
    return safe

pts1 = keep_safe(corners1, img.shape)
pts2 = keep_safe(corners2, img2.shape)

D1 = np.array([mops_descriptor(img, p, radius=20)[0] for p in pts1])
D2 = np.array([mops_descriptor(img2, p, radius=20)[0] for p in pts2])

matches = ratio_match(D1, D2, ratio=0.75)
print(f"Points in image 1: {len(pts1)}")
print(f"Points in image 2: {len(pts2)}")
print(f"Ratio-test matches: {len(matches)}")
print("First five matches as (index_in_img1, index_in_img2, nearest_distance, second_distance):")
print(matches[:5])

## 14. Capstone Mini-Project

Choose one of the following tasks and complete it in a new cell.

### Option A: Descriptor Robustness Study

Compare raw patches, normalized patches, MOPS, HOG, and SIFT-style descriptors under:

1. Brightness changes.
2. Rotation changes.
3. Blur changes.
4. Noise changes.

Report which descriptor is most stable for each transformation.

### Option B: Build a Feature-Matching Demo

Create two transformed versions of an image, detect Harris corners, compute descriptors, match them, and visualize the top matches.

### Option C: Descriptor Design Challenge

Design your own descriptor using any combination of:

- Patch normalization.
- Gradient magnitude and orientation.
- Spatial histograms.
- Integral-image rectangle sums.
- Gaussian weighting.

Explain what invariance your descriptor is intended to provide.

**Suggested evaluation metric**

For a descriptor $d$ computed before and after a transformation, measure relative change:

$$
\Delta(d, d') = \frac{\lVert d - d' \rVert_2}{\lVert d \rVert_2 + \epsilon}
$$

Lower values indicate more stable descriptors.

### Code Template: Descriptor Robustness Experiment

In [ ]:
# TODO: Complete this experiment.
# 1. Pick a safe keypoint center.
# 2. Compute descriptors before and after a chosen transformation.
# 3. Compare relative descriptor changes.

def relative_change(d, d2, eps=1e-8):
    """Measure relative descriptor change after a transformation."""
    return np.linalg.norm(d - d2) / (np.linalg.norm(d) + eps)

center = (128, 128)
base_patch = extract_patch(img, center, radius=24)
bright_img = np.clip(1.4 * img + 0.15, 0, 1)

raw1 = raw_descriptor(base_patch)
raw2 = raw_descriptor(extract_patch(bright_img, center, radius=24))

norm1 = normalized_patch_descriptor(base_patch)
norm2 = normalized_patch_descriptor(extract_patch(bright_img, center, radius=24))

mops1 = mops_descriptor(img, center, radius=20)[0]
mops2 = mops_descriptor(bright_img, center, radius=20)[0]

print("Relative change for raw pixels:", relative_change(raw1, raw2))
print("Relative change for normalized patch:", relative_change(norm1, norm2))
print("Relative change for MOPS:", relative_change(mops1, mops2))

# Extend this cell by adding HOG and SIFT-style comparisons.

## 15. Reflection Questions

Answer these questions in your own words:

1. Why is a descriptor usually more useful than a detector alone?
2. What is the main weakness of raw patch descriptors?
3. How does normalization help with photometric changes?
4. Why do spatial histograms preserve more structure than global histograms?
5. Why does HOG normalize over blocks rather than only cells?
6. What does SIFT gain by using orientation assignment and a $4 \times 4$ grid of gradient histograms?
7. What is the trade-off between discriminative power and generalization power?

A concise answer is better than a long answer if it clearly explains the idea.